In [7]:
import os

os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'  # MUST be before unsloth import

os.environ["LD_LIBRARY_PATH"] = (
    "/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)

print("ModelScope:", os.environ['UNSLOTH_USE_MODELSCOPE'])
print("LD set ✓")

ModelScope: 1
LD set ✓


In [6]:
!pip install modelscope -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 41.7 MB/s eta 0:00:00


In [5]:
# Install unsloth
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install trl==0.14.0 peft==0.14.0 -q
print("✅ Unsloth installed!")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.4/213.4 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.9/313.9 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 29.3 MB/s et

In [8]:
 #Step3: Import necessary libraries
import unsloth
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
from trl import SFTTrainer
from huggingface_hub import login
from transformers import TrainingArguments
from datasets import load_dataset
import wandb
print("✅ Imports done")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not find `steps_per_generation` in grpo_trainer
Unsloth: Could not find `generation_batch_size` in grpo_trainer
✅ Imports done


In [9]:
from google.colab import userdata
from huggingface_hub import login
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

In [10]:
# Optional: Checking GPU availability
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU device:", torch.cuda.get_device_name(0)) if torch.cuda.is_available() else "No GPU"


CUDA available: True
GPU device: Tesla T4


In [7]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Llama-8B",  # unsloth/ prefix for ModelScope
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
    token = hf_token,
)
print("✅ Model loaded!")

2026-06-16 16:37:37,364 - modelscope - INFO - Got 9 files, start to download ...


Processing 9 items:   0%|          | 0.00/9.00 [00:00<?, ?it/s]

2026-06-16 18:11:38,488 - modelscope - INFO - Finish downloading 9 files for repo 'unsloth/deepseek-r1-distill-llama-8b-unsloth-bnb-4bit'


==((====))==  Unsloth 2026.6.7: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load /root/.cache/modelscope/hub/models/unsloth/deepseek-r1-distill-llama-8b-unsloth-bnb-4bit as a legacy tokenizer.


✅ Model loaded!


In [11]:
# Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [22]:
# Cell 2 -
import shutil

tokenizer.save_pretrained("/content/drive/MyDrive/AI-Doctor-Model")

src = "/root/.cache/modelscope/hub/models/unsloth/deepseek-r1-distill-llama-8b-unsloth-bnb-4bit"
dst = "/content/drive/MyDrive/AI-Doctor-Model"

print("Copying model files... ⏳")
shutil.copytree(src, dst, dirs_exist_ok=True)
print("✅ Model Drive loaded!")

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/AI-Doctor-Model/tokenizer_config.json.


Copying model files... thoda time lagega ⏳
✅ Model Drive mein save ho gaya!


In [12]:
# Now I have to run this cell!
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/AI-Doctor-Model",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
print("✅ Loaded from Drive - no download!")

==((====))==  Unsloth 2026.6.7: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load /content/drive/MyDrive/AI-Doctor-Model as a legacy tokenizer.


✅ Loaded from Drive - no download!


In [13]:
#Setup system prompt
prompt_style = """
Below is a task description along with additional context provided in the input section. Your goal is to provide a well-reasoned response that effectively addresses the request.

Below crafting your answer, take a moment to carefully analyze the question. Develop a clear, step-by-step thought process to ensure your response in both and accurate.

### Instruction:
You are a medical expert specializing in clinical reasoning, diagnostics, and treatment planning. Answer the medical question below using your advanced knowledge.

### Question:
{}

### Response:
{}
"""

In [14]:
# Run Inference on the model

# Define a test question
question = """ A 61-year-old woman with a long history of involuntary urine loss duing activities like coughing or sneezing but no leakage at night undergoes a gynecological eaxm and Q-tip test. Based on these findings,what would cystometry most likely reveal about her residual volume and detrusor contractions.
"""

# Tokenize the input
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

# Generate a response
outputs = model.generate (
    input_ids = inputs.input_ids,
    attention_mask = inputs.attention_mask,
    max_new_tokens = 1200, # Corrected from max_new_token to max_new_tokens
    use_cache = True
)

# Decode the response tokens back to text
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)

clean = response[0]

clean = clean.replace("Ġ", " ")
clean = clean.replace("Ċ", "\n")

# Extract only the answer after <think>...</think>
if "</think>" in clean:
    clean = clean.split("</think>")[-1].strip()
elif "### Response:" in clean:
    clean = clean.split("### Response:")[-1].strip()

print(clean)


Both `max_new_tokens` (=1200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.

**Answer:**

1. **Understanding the Problem:**
   The patient is a 61-year-old woman with a history of involuntary urinary loss during activities like coughing or sneezing, but no leakage at night. She underwent a gynecological exam and a Q-tip test. The goal is to determine what cystometry (urodynamic testing) would reveal about her residual volume and detrusor contractions.

2. **Breaking Down the Symptoms:**
   The symptoms suggest a potential issue with the lower urinary tract, specifically the urethral sphincter's ability to close properly. The absence of leakage at night indicates normal bladder capacity and nocturnal emptying, ruling out storage or nocturnal issues.

3. **Q-Tip Test Interpretation:**
   The Q-tip test measures urethral sphincter function. A negative result (no resistance) indicates the sphincter does not close properly, leading to incontinence during activities like coughing or sneezing.

4. **Cystometry Findings:**
   Urodynamic evaluation would likely show nor

In [15]:
# setup fine-tuning

# Load Dataset
medical_dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT","en",split = "train[:500]", trust_remote_code = True)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'FreedomIntelligence/medical-o1-reasoning-SFT' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'FreedomIntelligence/medical-o1-reasoning-SFT' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

medical_o1_sft.json:   0%|          | 0.00/58.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19704 [00:00<?, ? examples/s]

In [16]:
medical_dataset[1]

{'Question': 'A 33-year-old woman is brought to the emergency department 15 minutes after being stabbed in the chest with a screwdriver. Given her vital signs of pulse 110/min, respirations 22/min, and blood pressure 90/65 mm Hg, along with the presence of a 5-cm deep stab wound at the upper border of the 8th rib in the left midaxillary line, which anatomical structure in her chest is most likely to be injured?',
 'Complex_CoT': "Okay, let's figure out what's going on here. A woman comes in with a stab wound from a screwdriver. It's in her chest, upper border of the 8th rib, left side, kind of around the midaxillary line. First thought, that's pretty close to where the lung sits, right?\n\nLet's talk about location first. This spot is along the left side of her body. Above the 8th rib, like that, is where a lot of important stuff lives, like the bottom part of the left lung, possibly the diaphragm too, especially considering how deep the screwdriver went.\n\nThe wound is 5 cm deep. Tha

In [17]:
EOS_TOKEN = tokenizer.eos_token
EOS_TOKEN
# Define EOS_TOKEN (End Of Sentence) which tells the model when to stop generating text during training

'<｜end▁of▁sentence｜>'

In [22]:
# Update the prompt_style for training

train_prompt_style = """
Below is a task description along with additional context provided in the input section. Your goal is to provide a well-reasoned response that effectively addresses the request.

Below crafting your answer, take a moment to carefully analyze the question. Develop a clear, step-by-step thought process to ensure your response in both and accurate.

### Instruction:
You are a medical expert specializing in clinical reasoning, diagnostics, and treatment planning. Answer the medical question below using your advanced knowledge.

### Question:
{}

### Response:
<think>
{}
</think>
{}
"""

In [23]:

# Prepare the data for fine-tuning

def preprocess_input_data(examples):
  inputs = examples["Question"]
  cots = examples["Complex_CoT"]
  outputs = examples["Response"]

  texts = []
  for input, cot, output in zip(inputs, cots, outputs):
    text = train_prompt_style.format(input, cot, output) + EOS_TOKEN
    texts.append(text)

  return {
      "texts" : texts,
  }

In [24]:
finetune_dataset = medical_dataset.map(
    preprocess_input_data,
    batched=True
)

In [25]:
finetune_dataset["texts"][1]

"\nBelow is a task description along with additional context provided in the input section. Your goal is to provide a well-reasoned response that effectively addresses the request.\n\nBelow crafting your answer, take a moment to carefully analyze the question. Develop a clear, step-by-step thought process to ensure your response in both and accurate.\n\n### Instruction:\nYou are a medical expert specializing in clinical reasoning, diagnostics, and treatment planning. Answer the medical question below using your advanced knowledge.\n\n### Question:\nA 33-year-old woman is brought to the emergency department 15 minutes after being stabbed in the chest with a screwdriver. Given her vital signs of pulse 110/min, respirations 22/min, and blood pressure 90/65 mm Hg, along with the presence of a 5-cm deep stab wound at the upper border of the 8th rib in the left midaxillary line, which anatomical structure in her chest is most likely to be injured?\n\n### Response:\n<think>\nOkay, let's figur

In [26]:
# Setup/Apply LoRA finetuning to the model
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r = 16,
    lora_alpha = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout = 0,
    bias = "none",
    task_type = "CAUSAL_LM",
)

model_lora = get_peft_model(model, peft_config)
model_lora.print_trainable_parameters()
print("LoRA ready!")

trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196
LoRA ready!


In [27]:
# Add this before creating the trainer
if hasattr(model, '_unwrapped_old_generate'):
  del model._unwrapped_old_generate

In [28]:
print(type(finetune_dataset["texts"][0]))
print(finetune_dataset["texts"][0][:500])

<class 'str'>

Below is a task description along with additional context provided in the input section. Your goal is to provide a well-reasoned response that effectively addresses the request.

Below crafting your answer, take a moment to carefully analyze the question. Develop a clear, step-by-step thought process to ensure your response in both and accurate.

### Instruction:
You are a medical expert specializing in clinical reasoning, diagnostics, and treatment planning. Answer the medical question below u


In [29]:
print(finetune_dataset["texts"][0][-1000:])

den weakness - a PFO fits the bill perfectly, letting a clot cross over and cause all this.

Everything fits together pretty neatly, so I'd bet PFO is the heart issue waiting to be discovered. Yeah, that really clicks into place!
</think>
The specific cardiac abnormality most likely to be found in this scenario is a patent foramen ovale (PFO). This condition could allow a blood clot from the venous system, such as one from a deep vein thrombosis in the leg, to bypass the lungs and pass directly into the arterial circulation. This can occur when the clot moves from the right atrium to the left atrium through the PFO. Once in the arterial system, the clot can travel to the brain, potentially causing an embolic stroke, which would explain the sudden weakness in the left arm and leg. The connection between the recent travel, which increases the risk of deep vein thrombosis, and the neurological symptoms suggests the presence of a PFO facilitating a paradoxical embolism.
<｜end▁of▁sentence｜>

In [3]:
!pip install trl -q
print("✅ TRL installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 22.6 MB/s eta 0:00:00
✅ TRL installed!


In [33]:
max_sequence_length = 2048

from trl import SFTTrainer, SFTConfig

# Dataset field name fix karo
finetune_dataset_fixed = finetune_dataset.map(
    lambda x: {"text": x["texts"][0] if isinstance(x["texts"], list) else x["texts"]},
    remove_columns=["texts"]
)

trainer = SFTTrainer(
    model = model_lora,
    tokenizer = tokenizer,
    train_dataset = finetune_dataset_fixed,
    dataset_text_field = "text",  # ← "texts" nahi, "text" hai
    args = SFTConfig(
        max_seq_length = max_sequence_length,
        dataset_num_proc = 1,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        gradient_checkpointing = False,
        save_strategy = "no",
        report_to = "none",
    )
)
print("✅ Trainer ready!")

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

/content/unsloth_compiled_cache/UnslothSFTTrainer.py:1070: UserWarning: You passed `remove_unused_columns=False` on a non-packed dataset. This might create some issues with the default collator and yield to errors. If you want to inspect dataset other columns (in this case ['Response', 'Complex_CoT', 'Question', 'text']), you can subclass `DataCollatorForLanguageModeling` in case you used the default collator and create your own data collator in order to inspect the unused dataset columns.
  warnings.warn(


Map (num_proc=1):   0%|          | 0/500 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
✅ Trainer ready!


In [34]:
# Setup, WANDB
from google.colab import userdata
wnb_token = userdata.get("WANDB_API_TOKEN")
#login to WnB
wandb.login(key=wnb_token) #import wandb
run = wandb.init(
    project='Fine-tune-DeepSeek-R1-on-Medical-CoT-Dataset',
    job_type="training",
    anonymous="allow"
)


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: prashantrck1998 (prashantrck1998-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


In [35]:
# Fix PicklingError - disable checkpoint saving
import os

# Training already completed! Just save the final model directly
model_lora.save_pretrained("outputs/final_model")
tokenizer.save_pretrained("outputs/final_model")
print("✅ Model saved successfully!")

Unsloth: Restored added_tokens_decoder metadata in outputs/final_model/tokenizer_config.json.


✅ Model saved successfully!


In [36]:
wandb.finish()

In [38]:
# Testing after finetuning
question = """ A 61-year-old woman with a long history of involuntary urine loss duing activities like coughing or sneezing but no leakage at night undergoes a gynecological eaxm and Q-tip test. Based on these findings,what would cystometry most likely reveal about her residual volume and detrusor contractions.
"""
# Ensure FastLanguageModel is defined in this scope
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model_lora)

# Tokenize the input
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

# Generate a response
outputs = model_lora.generate (
    input_ids = inputs.input_ids,
    attention_mask = inputs.attention_mask,
    max_new_tokens = 1200, # Corrected from max_new_token to max_new_tokens
    use_cache = True
)

# Decode the response tokens back to text
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
response = response.replace("Ġ", " ").replace("Ċ", "\n")
if "</think>" in response:
    response = response.split("</think>")[-1].strip()

print(response)

Both `max_new_tokens` (=1200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


**Answer:**

Based on the analysis, the cystometry would most likely reveal normal residual volume and normal detrusor contractions. 

1. **Residual Volume:** Since there is no leakage at night, the residual volume after voiding is likely within the normal range. If there were significant residual volume, it would typically cause nighttime leakage.

2. **Detrusor Contractions:** The detrusor is the muscle layer of the bladder. In this case, the primary issue appears to be related to sphincter function rather than bladder contractions. Therefore, the detrusor contractions are likely normal.

3. **Q-Tip Test Findings:** The Q-tip test would likely show low opening pressure, indicating urethral sphincter insufficiency, which contributes to the involuntary urinary loss during activities like coughing.

In summary, the findings from cystometry would indicate normal residual volume and normal detrusor contractions, with the primary issue likely related to sphincter function.
